# DB 검색 조회 4차 — 동점 기준을 넣고 결과가 fp32와 같은지 확인 (Colab)

3차에서 `halfvec 상위 100 → fp32 상위 50`(`rr`)과 법정동 코드 순 적재(`_sorted`)가 가장 빨랐습니다. 다만 **같은 fp32인데 넣는 순서만 바꾼 표(`plain_sorted`)도 결과 순서가 달랐습니다.** 다른 자리의 점수가 모두 같았으므로, 체인점처럼 문서가 같은 가게가 **동점**이고 쿼리에 동점 기준이 없어 순서가 저장 순서를 따른 것입니다. 정책 7-5(같은 점수면 `store_id` 오름차순, `docs/search/SEARCH_HANDOFF.md` 2절 10)를 쿼리에 넣고 다시 비교합니다.

| 이름 | 표 | 쿼리(동점이면 store_id 오름차순) |
|---|---|---|
| `plain_sorted` | fp32 `STORAGE PLAIN`, 법정동 코드 순(기준) | fp32 상위 50 |
| `rr_sorted` | halfvec PLAIN + fp32 TOAST, 법정동 코드 순 | halfvec 상위 100 → fp32 상위 50 |
| `base` | 지금 파티션(fp32 TOAST, 원래 순서) | fp32 상위 50. 반경 10km·시군구만, 저장 방식·순서와 결과가 무관한지 확인용 |

**통과 조건**: 모든 경우·모든 질의에서 상위 50의 가게와 순서가 `plain_sorted`와 완전히 같음.

**Drive에 둘 것**: 1~3차와 같습니다. **결과**: `service_load/bench/db_tiebreak_<시각>.md`.

- 런타임: GPU(T4). 전체 약 25분(설치·복원 8분, 표 두 개 7분, 측정·비교 10분). 동시 부하는 이번엔 재지 않습니다.

In [ ]:
import os, subprocess, multiprocessing
print("CPU", multiprocessing.cpu_count(), "개")
print(subprocess.run("free -g | head -2; df -h /tmp | tail -1; nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%bash
# PostgreSQL 16 + PostGIS 3 + pgvector (PGDG 저장소). 1차와 같은 설정
set -e
apt-get -qq update > /dev/null
apt-get -qq install -y curl ca-certificates lsb-release > /dev/null
install -d /usr/share/postgresql-common/pgdg
curl -s -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.asc https://www.postgresql.org/media/keys/ACCC4CF8.asc
echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.asc] https://apt.postgresql.org/pub/repos/apt $(lsb_release -cs)-pgdg main" > /etc/apt/sources.list.d/pgdg.list
apt-get -qq update > /dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y postgresql-16 postgresql-16-postgis-3 postgresql-16-pgvector > /dev/null
pg_ctlcluster 16 main start || true
runuser -u postgres -- psql -qc "ALTER SYSTEM SET shared_buffers = '2GB'" -c "ALTER SYSTEM SET work_mem = '64MB'" \
  -c "ALTER SYSTEM SET maintenance_work_mem = '1GB'" -c "ALTER USER postgres PASSWORD 'bench'"
pg_ctlcluster 16 main restart
runuser -u postgres -- psql -tAc "select version()"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/store-search-ai")
DUMP = ROOT / "service_load/storesearch_v1.dump"
QUERIES = ROOT / "service_load/bench/service_queries_v1.csv"
MODEL_DIR = ROOT / "runs/finetune/bge_m3_ft_mj_20261014_0743"
OUT_DIR = ROOT / "service_load/bench"
for p in [DUMP, QUERIES, MODEL_DIR]:
    assert p.exists(), f"Drive에 없음: {p}"
print(f"덤프 {DUMP.stat().st_size / 1e9:.2f}GB")

In [ ]:
%%bash
# Drive에서 로컬 디스크로 복사한 뒤 복원
set -e
cp "/content/drive/MyDrive/store-search-ai/service_load/storesearch_v1.dump" /tmp/storesearch_v1.dump
chmod 644 /tmp/storesearch_v1.dump
runuser -u postgres -- dropdb --if-exists storesearch
runuser -u postgres -- createdb storesearch
runuser -u postgres -- pg_restore -d storesearch --no-owner -j 2 /tmp/storesearch_v1.dump 2> /tmp/restore.err || true
echo "복원 경고·오류 $(grep -c ERROR /tmp/restore.err || true)건(public 스키마 등 이미 있는 것은 무시해도 됨)"; grep ERROR /tmp/restore.err | head -5 || true
runuser -u postgres -- psql -d storesearch -tAc "select version_name, status, n_total from data_version; select count(*) from store_snapshot"
rm /tmp/storesearch_v1.dump

In [ ]:
!pip install -q "psycopg[binary,pool]" sentence-transformers==5.7.0 transformers==5.18.0

In [ ]:
# 질의 벡터: 임베딩 서버와 같은 설정(fp32, 128토큰, 정규화)
import numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

qs = pd.read_csv(QUERIES, dtype=str, keep_default_na=False, encoding="utf-8-sig")
qs = qs[(qs["status"] == "active") & (qs["latitude"] != "") & (qs["longitude"] != "")].reset_index(drop=True)
qs["lat"] = qs["latitude"].astype(float)
qs["lng"] = qs["longitude"].astype(float)
model = SentenceTransformer(str(MODEL_DIR), device="cuda", model_kwargs={"dtype": torch.float32})
model.max_seq_length = 128
qv = model.encode(qs["query"].tolist(), batch_size=128, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
qs["vec"] = ["[" + ",".join(f"{x:.8g}" for x in v) + "]" for v in qv]
del model
torch.cuda.empty_cache()
print(len(qs), "개 질의")

In [ ]:
import re, time, psycopg
DSN = "host=127.0.0.1 dbname=storesearch user=postgres password=bench"
conn = psycopg.connect(DSN, autocommit=True)
conn.execute("create extension if not exists pg_prewarm")
V = conn.execute("select version_id from data_version where status = 'active'").fetchone()[0]
BASE = f"store_snapshot_v{V}"
settings = {k: conn.execute(f"show {k}").fetchone()[0] for k in ["shared_buffers", "work_mem", "max_parallel_workers_per_gather"]}
pg = conn.execute("select extversion from pg_extension where extname = 'vector'").fetchone()[0]
print("버전", V, BASE, settings, "pgvector", pg)

# 질의 위치에서 가장 가까운 가게의 법정동 코드(② 시군구·시도 범위용)
qs["code"] = [conn.execute(
    f"select legal_dong_code from {BASE} where searchable "
    "order by geom <-> ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography limit 1", (r.lng, r.lat)).fetchone()[0]
    for r in qs.itertuples()]

In [ ]:
# 비교용 표 두 개(법정동 코드 순). 기존 파티션을 복사하되 벡터 저장과 넣는 순서만 바꾼다
cols = [c for (c,) in conn.execute(
    "select column_name from information_schema.columns where table_name = %s order by ordinal_position", (BASE,)).fetchall()]
index_defs = [d for (d,) in conn.execute("select indexdef from pg_indexes where tablename = %s", (BASE,)).fetchall()]


def build(table, with_half):
    t = time.perf_counter()
    conn.execute(f"drop table if exists {table}")
    conn.execute(f"create table {table} (like {BASE} including defaults)")
    names = [f'"{c}"' for c in cols]
    sel = [f'"{c}"' for c in cols]
    if with_half:
        conn.execute(f"alter table {table} add column embedding_half halfvec(1024) not null")
        conn.execute(f"alter table {table} alter column embedding_half set storage plain")
        names.append("embedding_half")
        sel.append("embedding::halfvec(1024)")
    else:
        conn.execute(f"alter table {table} alter column embedding set storage plain")
    conn.execute(f"insert into {table} ({', '.join(names)}) select {', '.join(sel)} from {BASE} order by legal_dong_code, store_id")
    for i, d in enumerate(index_defs):
        conn.execute(re.sub(r"INDEX \S+ ON (ONLY )?\S+", f"INDEX {table}_ix{i} ON public.{table}", d))
    conn.execute(f"vacuum analyze {table}")
    print(table, f"{time.perf_counter() - t:.0f}s")


build("bench_plain_sorted", False)
build("bench_rr_sorted", True)

In [ ]:
# 동점이면 store_id 오름차순(정책 7-5). 다시 정렬하는 쪽은 안쪽(halfvec)·바깥쪽(fp32) 둘 다 같은 기준
RERANK_K = 100
WHERE = "from {T} s where s.version_id = %(ver)s and s.searchable "
EXACT = ("select s.store_id, 1 - (s.embedding <=> %(v)s::vector) as cosine " + WHERE + "{COND}"
         " order by s.embedding <=> %(v)s::vector, s.store_id limit 50")
RERANK = ("select c.store_id, 1 - (c.embedding <=> %(v)s::vector) as cosine from ("
          "select s.store_id, s.embedding " + WHERE + "{COND}"
          f" order by s.embedding_half <=> %(v)s::halfvec, s.store_id limit {RERANK_K}"
          ") c order by c.embedding <=> %(v)s::vector, c.store_id limit 50")
RADIUS = "and ST_DWithin(s.geom, ST_SetSRID(ST_MakePoint(%(lng)s, %(lat)s), 4326)::geography, %(r)s)"
PREFIX = "and s.legal_dong_code like %(p)s || '%%'"
BOX = "and s.geom && ST_MakeEnvelope(%(w)s, %(s)s, %(e)s, %(n)s, 4326)::geography"
CITY, WIDE = (0.03, 0.02), (0.15, 0.10)


def box(r, size):
    return {"w": r.lng - size[0], "s": r.lat - size[1], "e": r.lng + size[0], "n": r.lat + size[1]}


CASES = [
    ("① 반경 1km", RADIUS, lambda r: {"r": 1000}),
    ("① 반경 10km", RADIUS, lambda r: {"r": 10000}),
    ("② 시군구", PREFIX, lambda r: {"p": r.code[:5]}),
    ("② 시도", PREFIX, lambda r: {"p": r.code[:2]}),
    ("③ 지도 도시 화면", BOX, lambda r: box(r, CITY)),
    ("③ 지도 넓은 화면", BOX, lambda r: box(r, WIDE)),
]
VARIANTS = [("plain_sorted", "bench_plain_sorted", False), ("rr_sorted", "bench_rr_sorted", True)]


def sql_for(cond, table, rerank):
    return (RERANK if rerank else EXACT).replace("{T}", table).replace("{COND}", cond)


def params(r, extra):
    return {"v": r.vec, "ver": V, "lng": r.lng, "lat": r.lat, **extra(r)}


def pct(a, q):
    return float(np.percentile(a, q, method="higher"))


def fmt(x):
    return f"{x:.1f}"


def prewarm(table):
    rels = conn.execute(
        "select c.oid from pg_class c where c.relname = %(t)s "
        "union all select c.reltoastrelid from pg_class c where c.relname = %(t)s and c.reltoastrelid <> 0",
        {"t": table}).fetchall()
    oids = [o for (o,) in rels]
    oids += [i for (i,) in conn.execute("select indexrelid from pg_index where indrelid = any(%s)", (oids,)).fetchall()]
    for oid in oids:
        conn.execute("select pg_prewarm(%s::oid::regclass)", (oid,))


def run(table, rerank, cond, extra):
    # 질의마다 (시간 ms, 결과 행)
    q = sql_for(cond, table, rerank)
    out = []
    for r in qs.itertuples():
        t = time.perf_counter()
        rows_ = conn.execute(q, params(r, extra)).fetchall()
        out.append(((time.perf_counter() - t) * 1000, rows_))
    return out

In [ ]:
# 표마다 메모리에 올린 뒤: 한 번 데우고(버림) 한 번 잰다. 잰 회차의 결과로 비교한다
results, rows_by = {}, {}
for name, table, rerank in VARIANTS:
    prewarm(table)
    for case, cond, extra in CASES:
        run(table, rerank, cond, extra)
        got = run(table, rerank, cond, extra)
        results[(name, case)] = [ms for ms, _ in got]
        rows_by[(name, case)] = [rs for _, rs in got]
        ms = results[(name, case)]
        print(f"{name:12} {case:12} p50 {fmt(pct(ms, 50)):>7} p95 {fmt(pct(ms, 95)):>7}")

In [ ]:
# 결과 비교: plain_sorted(기준)와 가게·순서가 완전히 같은 질의 수. 다르면 예시를 남긴다
def compare(ref, got, label, case, diffs):
    same = 0
    for qi, (a, b) in enumerate(zip(ref, got)):
        ida, idb = [x[0] for x in a], [x[0] for x in b]
        if ida == idb:
            same += 1
            continue
        if len(diffs) < 20:
            m = min(len(ida), len(idb))
            k = next((i for i in range(m) if ida[i] != idb[i]), m)
            sa = f"{a[k][1]:.7f}" if k < len(a) else "-"
            sb = f"{b[k][1]:.7f}" if k < len(b) else "-"
            diffs.append(f"| {label} | {case} | {qs['query'].iloc[qi]} | {k + 1} | {sa} | {sb} | {len(a)} / {len(b)} |")
    return same


agree, diffs = [], []
n = len(qs)
for case, _, _ in CASES:
    same = compare(rows_by[("plain_sorted", case)], rows_by[("rr_sorted", case)], "rr_sorted", case, diffs)
    agree.append(f"| {case} | rr_sorted | {same}/{n} |")

# 저장 방식·순서와 결과가 무관한지: 지금 파티션(fp32 TOAST, 원래 순서)과 같은 쿼리로
prewarm(BASE)
for case, cond, extra in [CASES[1], CASES[2]]:
    base_rows = [rs for _, rs in run(BASE, False, cond, extra)]
    same = compare(rows_by[("plain_sorted", case)], base_rows, "base", case, diffs)
    agree.append(f"| {case} | base | {same}/{n} |")
print("\n".join(agree))
print("\n".join(diffs) or "다른 질의 없음")

# 동점이 실제로 얼마나 있나: 기준 결과 안에서 바로 앞 가게와 점수가 같은 자리 수
ties = []
for case, _, _ in CASES:
    rs = rows_by[("plain_sorted", case)]
    tied_q = sum(any(abs(r_[i][1] - r_[i - 1][1]) < 1e-9 for i in range(1, len(r_))) for r_ in rs)
    ties.append(f"| {case} | {tied_q}/{n} |")
print("\n".join(ties))

In [ ]:
# 가장 무거운 경우(같은 쿼리 30번, 첫 번 제외)와 시도 전체 EXPLAIN
v = qs["vec"].iloc[0]
sido, sido_n = conn.execute(f"select left(legal_dong_code, 2), count(*) from {BASE} where searchable "
                            "group by 1 order by 2 desc limit 1").fetchone()
lng, lat, dense_n = conn.execute(
    f"select ST_X(geom::geometry), ST_Y(geom::geometry), (select count(*) from {BASE} t where t.searchable "
    f"and ST_DWithin(t.geom, s.geom, 10000)) as n from {BASE} s tablesample system (1) "
    "where searchable and legal_dong_code like '11%%' order by n desc limit 1").fetchone()
heavy_rows, plans = [], []
for name, table, rerank in VARIANTS:
    prewarm(table)
    cases = [(f"시도 전체(코드 {sido}, {sido_n}곳)", sql_for(PREFIX, table, rerank), {"v": v, "ver": V, "p": sido}),
             (f"서울 가장 빽빽한 곳 10km({dense_n}곳)", sql_for(RADIUS, table, rerank),
              {"v": v, "ver": V, "lng": lng, "lat": lat, "r": 10000})]
    for case, q, p in cases:
        conn.execute(q, p).fetchall()
        ms = []
        for _ in range(30):
            t = time.perf_counter()
            conn.execute(q, p).fetchall()
            ms.append((time.perf_counter() - t) * 1000)
        heavy_rows.append(f"| {name} | {case} | {fmt(pct(ms, 50))} | {fmt(pct(ms, 95))} | {fmt(max(ms))} |")
    plan = conn.execute("explain (analyze, buffers, costs off) " + cases[0][1], cases[0][2]).fetchall()
    plans.append(f"-- {name}\n" + "\n".join(re.sub(r"'\[[^']*\]'", "'[…]'", row[0]) for row in plan))
print("\n".join(heavy_rows))

In [ ]:
names = [n_ for n_, _, _ in VARIANTS]
report = [f"# DB 검색 조회 4차: 동점 기준(store_id) 넣고 fp32와 결과 비교 — Colab ({time.strftime('%Y-%m-%d %H:%M')} UTC)\n",
          f"- 환경: Colab CPU {multiprocessing.cpu_count()}개, PostgreSQL 16, pgvector {pg}, shared_buffers {settings['shared_buffers']}, "
          f"work_mem {settings['work_mem']}, max_parallel_workers_per_gather {settings['max_parallel_workers_per_gather']}",
          f"- 데이터 버전 {V}, 질의 {len(qs)}개. 표마다 pg_prewarm → 한 번 데우고 → 한 번 잼(ms)",
          f"- plain_sorted = fp32 PLAIN, rr_sorted = halfvec PLAIN + fp32 TOAST(halfvec 상위 {RERANK_K} → fp32 상위 50). 둘 다 법정동 코드 순으로 넣음. 쿼리는 동점이면 store_id 오름차순\n",
          "## 결과가 plain_sorted와 완전히 같은 질의 수(상위 50 가게·순서)\n",
          "| 경우 | 표 | 같음 |\n|---|---|---|", *agree,
          "\n### 다른 경우 예시(최대 20개)\n",
          "| 표 | 경우 | 질의 | 처음 다른 순위 | 기준 점수 | 표 점수 | 결과 수 |\n|---|---|---|---|---|---|---|", *(diffs or ["| - | - | 없음 | - | - | - | - |"]),
          "\n## 동점이 있는 질의 수(기준 결과 안에서 바로 앞 가게와 점수가 같은 자리가 있음)\n",
          "| 경우 | 질의 |\n|---|---|", *ties,
          "\n## p95(ms) — 기준 150\n\n| 경우 | " + " | ".join(names) + " |\n|---|" + "---|" * len(names)]
for case, _, _ in CASES:
    report.append(f"| {case} | " + " | ".join(fmt(pct(results[(n_, case)], 95)) for n_ in names) + " |")
report.append("\n## p50(ms)\n\n| 경우 | " + " | ".join(names) + " |\n|---|" + "---|" * len(names))
for case, _, _ in CASES:
    report.append(f"| {case} | " + " | ".join(fmt(pct(results[(n_, case)], 50)) for n_ in names) + " |")
report += [f"\n## 가장 무거운 경우(질의 '{qs['query'].iloc[0]}', 30번)\n",
           "| 표 | 경우 | p50 | p95 | 최대 |\n|---|---|---|---|---|", *heavy_rows,
           "\n## 시도 전체 EXPLAIN\n\n```\n" + "\n\n".join(plans) + "\n```"]
out = OUT_DIR / f"db_tiebreak_{time.strftime('%Y%m%d_%H%M')}.md"
out.write_text("\n".join(report) + "\n", encoding="utf-8")
print(out)
print("\n".join(report))